# Section 5: input steering length and prompt and unsteered baselines

Fixed replay, layer 17 / head 0, prediction step 128. All points and the prompt and unsteered baselines use the same 100 behaviors with at least 64 input tokens. Shaded regions are 95% bootstrap confidence intervals across behaviors.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "results/section5_fixed_summary.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/ directory")
summary = json.loads((ROOT / "results/section5_fixed_summary.json").read_text())
series = sorted(
    (row for row in summary["input_length_matched_cohort"] if row["alpha"] == 1),
    key=lambda row: row["k"] if row["k"] > 0 else float("inf"),
)
assert [row["k"] for row in series] == [1, 4, 16, 64, -1]
assert {row["n"] for row in series} == {100}

cohort = {"prompt": [], "unsteered": []}
for path in sorted((ROOT / "results").glob("section5_fixed_[0-9][0-9][0-9].json")):
    state = json.loads(path.read_text())
    unit = state["units"]["17_0"]
    if unit["base_length"] >= 64:
        for method, condition in (("prompt", "prompt_m8"), ("unsteered", "unsteered")):
            row = next(row for row in unit["rows"] if row["id"] == condition)
            assert row["eligible"]
            cohort[method].append(row)
assert all(len(rows) == 100 for rows in cohort.values())

baseline = {}
for method, rows in cohort.items():
    baseline[method] = {}
    for field in ("R", "C"):
        values = np.array([row[field] for row in rows if row[field] is not None])
        assert len(values) == 100
        draws = np.random.default_rng(42).integers(0, len(values), size=(2000, len(values)))
        lower, upper = np.quantile(values[draws].mean(axis=1), [.025, .975])
        baseline[method][field] = {"mean": float(values.mean()), "lower": float(lower), "upper": float(upper)}

baseline

In [ ]:
# Steering by input-token count: sample std and 95% bootstrap CI over the matched 100 behaviors.
steering = {1: {'R': {'mean': 0.999910438195332,
           'std': 0.00032677588461285043,
           'lower': 0.999836590887384,
           'upper': 0.9999524302271202},
     'C': {'mean': 0.17746237142548257,
           'std': 0.16541056800072854,
           'lower': 0.14779944285743987,
           'upper': 0.21332975460941664}},
 4: {'R': {'mean': 0.9987069876830944,
           'std': 0.003029845540377656,
           'lower': 0.9980323117638759,
           'upper': 0.9992034883783062},
     'C': {'mean': 0.2012688507810665,
           'std': 0.16250222944427017,
           'lower': 0.1724965980114044,
           'upper': 0.23574817091511102}},
 16: {'R': {'mean': 0.9851533199579714,
            'std': 0.02163300690025975,
            'lower': 0.980802151170088,
            'upper': 0.9888634174770788},
      'C': {'mean': 0.2879609465255586,
            'std': 0.16988942278010594,
            'lower': 0.2579289737681327,
            'upper': 0.3226960006270176}},
 64: {'R': {'mean': 0.9169053552318416,
            'std': 0.09635015000991548,
            'lower': 0.8980042511542928,
            'upper': 0.9342934780584633},
      'C': {'mean': 0.45462550067739477,
            'std': 0.23141422575602896,
            'lower': 0.4119615054255502,
            'upper': 0.5014200172293924}},
 'Full': {'R': {'mean': 0.69777660683872,
                'std': 0.2483549823123861,
                'lower': 0.6507485355042291,
                'upper': 0.7457203890806362},
          'C': {'mean': 0.7074133878358997,
                'std': 0.29020031916558064,
                'lower': 0.6519724114165163,
                'upper': 0.7646309219198419}}}

steering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
x = np.arange(len(steering))
labels = [str(k) for k in steering]
for ax, field in zip(axes, ("R", "C")):
    mean = [steering[k][field]["mean"] for k in steering]
    lower = [steering[k][field]["lower"] for k in steering]
    upper = [steering[k][field]["upper"] for k in steering]
    ax.plot(x, mean, "o-", color="tab:blue", label="Steering")
    ax.fill_between(x, lower, upper, color="tab:blue", alpha=.15)
    ax.axhline(baseline["prompt"][field]["mean"], color="tab:green", linestyle="--", label="Prompt (m=8)")
    ax.axhspan(baseline["prompt"][field]["lower"], baseline["prompt"][field]["upper"], color="tab:green", alpha=.12)
    ax.axhline(baseline["unsteered"][field]["mean"], color="black", linestyle=":", label="Unsteered")
    ax.axhspan(baseline["unsteered"][field]["lower"], baseline["unsteered"][field]["upper"], color="black", alpha=.07)
    ax.set_xticks(x, labels)
    ax.set_xlabel("Steered input tokens (m=8, α=1)")
    ax.set_ylabel(field)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
axes[0].set_ylim(top=1.015)
fig.suptitle("Matched cohort: n=100")
output = ROOT / "figs/section5_input_length.pdf"
fig.savefig(output)
plt.show()